In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler,OneHotEncoder, FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import cross_val_score, train_test_split, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline, make_pipeline
from category_encoders import TargetEncoder
import statsmodels.api as sm
from sklearn.model_selection import learning_curve
from sklearn.metrics import roc_curve, precision_recall_curve, auc, precision_score, recall_score, f1_score, roc_auc_score
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import shap
import tensorflow as tf
from sklearn.utils.class_weight import compute_class_weight
from functools import partial
import joblib
from pytorch_tabnet.tab_model import TabNetClassifier
import optuna
import tensorflow as tf
import keras
import torch

from sklearn import set_config
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)

pretrained bert model, oversampling techniques....  k means t find new features? clustering methods? Use a faster training model then than TabNet probably...

Initialize

In [2]:
df = pd.read_csv('..\Data\loan_data_sample.csv')
features = ['loan_amnt', 'term', 'emp_length', 'home_ownership',
       'annual_inc', 'verification_status', 'purpose', 'dti', 'delinq_2yrs',
       'inq_last_6mths', 'mths_since_last_delinq', 'mths_since_last_record',
       'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
       'initial_list_status', 'mths_since_last_major_derog',
       'application_type', 'tot_coll_amt', 'tot_cur_bal', 'open_acc_6m',
       'open_act_il', 'open_il_12m', 'open_il_24m', 'mths_since_rcnt_il',
       'total_bal_il', 'il_util', 'open_rv_12m', 'open_rv_24m', 'max_bal_bc',
       'all_util', 'total_rev_hi_lim', 'inq_fi', 'total_cu_tl', 'inq_last_12m',
       'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util',
       'chargeoff_within_12_mths', 'mo_sin_old_il_acct',
       'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl',
       'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq',
       'mths_since_recent_inq', 'mths_since_recent_revol_delinq',
       'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl',
       'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
       'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m',
       'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
       'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies',
       'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit',
       'total_il_high_credit_limit',
       'months_sincefrst_credit', 'public_record', 'is_consolidation',
       'addr_state', 'is_currently_delinq', 'has_il_history']

index_sql = 'Loan_ID'
target = 'predictor'

df_features  = df[features]
df_predictor = pd.Series(df[target])


#Imputing col's we imputed with 999 in SQL
imputed_cols = [
    'mths_since_last_delinq', 'mths_since_last_record', 
    'mths_since_last_major_derog', 'mths_since_recent_bc_dlq', 
    'mths_since_recent_inq', 'mths_since_recent_revol_delinq'
]

df_features.loc[:,imputed_cols] = df_features[imputed_cols].replace(999.0, np.nan)





X_train, X_test, y_train,y_test = train_test_split(df_features,df_predictor,stratify=df_predictor,test_size=.2,random_state=11)

categorical_features = X_train.select_dtypes(include=['object','category']).columns.tolist()
numerical_features = X_train.select_dtypes(include=['number']).columns.tolist()

Pipeline for Flags / missing values  --- Target Encoding

In [3]:

zero_cols = [
    'max_bal_bc', 'all_util', 'il_util', 'open_acc_6m',
    'open_il_12m', 'open_il_24m', 'open_rv_12m', 'open_rv_24m', 'inq_last_12m',
    'open_act_il', 'total_bal_il', 'total_il_high_credit_limit', 'is_consolidation'
]
flag_cols = [
    'mths_since_last_delinq', 'mths_since_last_record',
    'mths_since_recent_bc_dlq', 'mths_since_recent_revol_delinq',
    'mths_since_recent_inq', 'mths_since_rcnt_il',
    'mths_since_last_major_derog'
]
median_cols = [
    'months_sincefrst_credit', 'annual_inc', 'inq_last_6mths',
    'revol_util', 'total_acc', 'pub_rec', 'open_acc',
    'mo_sin_old_rev_tl_op', 'num_rev_accts', 'tot_hi_cred_lim',
    'acc_open_past_24mths', 'num_bc_sats', 'num_sats', 'mort_acc',
    'mths_since_recent_bc', 'total_bc_limit', 'pub_rec_bankruptcies',
    'total_rev_hi_lim', 'inq_fi', 'avg_cur_bal', 'bc_open_to_buy',
    'bc_util', 'mo_sin_old_il_acct', 'mo_sin_rcnt_rev_tl_op',
    'mo_sin_rcnt_tl', 'num_accts_ever_120_pd', 'num_actv_bc_tl',
    'num_actv_rev_tl', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl',
    'num_rev_tl_bal_gt_0', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m',
    'pct_tl_nvr_dlq', 'percent_bc_gt_75',
    'total_cu_tl', 'total_bal_ex_mort', 'num_tl_30dpd',
    'num_tl_120dpd_2m', 'chargeoff_within_12_mths'
]
ohe_cols   = ['home_ownership', 'verification_status', 'application_type']
target_col = ['addr_state']


def winsorize_df(X):
    if hasattr(X, 'columns'):
        return X.clip(lower=X.quantile(0.01), upper=X.quantile(0.99), axis=1)
    return X

def freecashflow(X):
    return (X[:, [0]] / 12) * (1 - (X[:, [1]] / 100))

def freecash_name(function_transformer, feature_names_in):
    return ['FE_free_cash_flow']

def make_ratio(X):
    eps = 0.001
    return X[:, [0]] / (X[:, [1]] + eps)

def monthlycash(X):
    return (X[:, [0]] / 12) * (1 - (X[:, [1]] / 100))

def ratio_name(function_transformer, feature_names_in):
    return ['custom_ratio']

def ratio_pipe():
    return Pipeline([
        ('impute',    SimpleImputer(strategy='median')),
        ('ratio',     FunctionTransformer(make_ratio, feature_names_out=ratio_name)),
    ])

zero_pipe = Pipeline([
    ('impute',    SimpleImputer(strategy='constant', fill_value=0))
])

median_pipe = Pipeline([
    ('impute',    SimpleImputer(strategy='median'))
])

flag_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median', add_indicator=True))
])

cat_ohe_pipe = Pipeline([
    ('impute',  SimpleImputer(strategy='most_frequent')),
    ('onehot',  OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=False))
])

target_enc_pipe = Pipeline([
    ('target_encode', TargetEncoder(smoothing=10, handle_missing='value', handle_unknown='value'))
])

monthly_pipe = Pipeline([
    ('impute',    SimpleImputer(strategy='median')),
    ('calc',      FunctionTransformer(monthlycash, feature_names_out=ratio_name))
])

freecash_pipe = Pipeline([
    ('impute', SimpleImputer(strategy='median')),
    ('calc',   FunctionTransformer(freecashflow, feature_names_out=freecash_name))
])

set_config(transform_output="default")

preprocessor = ColumnTransformer(
    transformers=[
        ('zeros',            zero_pipe,        zero_cols),
        ('medians',          median_pipe,       median_cols),
        ('flags',            flag_pipe,         flag_cols),
        ('ohe',              cat_ohe_pipe,      ohe_cols),
        ('target',           target_enc_pipe,   target_col),
        ('FE_free_cash_flow',freecash_pipe, ['annual_inc', 'dti']),
        ('FE_loan_to_income',ratio_pipe(),      ['loan_amnt', 'annual_inc']),
        ('FE_activity_ratio',ratio_pipe(),      ['num_actv_rev_tl', 'num_op_rev_tl']),
        ('my_monthly_cash',  monthly_pipe,      ['annual_inc', 'dti'])
    ],
    remainder='drop'
)



In [4]:
X_train_pre = preprocessor.fit_transform(X_train, y_train)
X_test_pre = preprocessor.transform(X_test)

# convert to pandas
X_train_pre = pd.DataFrame(X_train_pre, columns=preprocessor.get_feature_names_out())
X_test_pre = pd.DataFrame(X_test_pre, columns=preprocessor.get_feature_names_out())

#joblib.dump(preprocessor,'../models/preprocessor_ANN.pkl')

## Fine Tuning

In [5]:
def objective_tabnet(trial):
    params = {
        'n_d': trial.suggest_int('n_d', 8, 64, step=8),
        'n_a': trial.suggest_int('n_a', 8, 64, step=8),
        'n_steps': trial.suggest_int('n_steps', 2, 8),
        'gamma': trial.suggest_float('gamma', 1.0, 2.0),
        'lambda_sparse': trial.suggest_float('lambda_sparse', 1e-6, 1e-2, log=True),
        'mask_type': trial.suggest_categorical('mask_type', ['sparsemax', 'entmax']),
        'seed': 11,
        'verbose': 0
    }

    lr = trial.suggest_float('lr', 1e-3, 1e-1, log=True)
    batch_size = trial.suggest_categorical('batch_size', [256, 512, 1024])

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=11)
    auc_scores = []

    for train_idx, val_idx in cv.split(X_train_pre, y_train):
        X_tr  = X_train_pre.iloc[train_idx].values
        y_tr  = y_train.iloc[train_idx].values
        X_val = X_train_pre.iloc[val_idx].values
        y_val = y_train.iloc[val_idx].values

        clf = TabNetClassifier(
            **params,
            optimizer_fn=torch.optim.Adam,
            optimizer_params=dict(lr=lr),
        )

        clf.fit(
            X_tr, y_tr,
            eval_set=[(X_val, y_val)],
            eval_metric=['auc'],
            max_epochs=100,
            patience=20,
            batch_size=batch_size,
            virtual_batch_size=batch_size // 4,
            weights=1  # handles class imbalance automatically
        )

        preds = clf.predict_proba(X_val)[:, 1]
        auc_scores.append(roc_auc_score(y_val, preds))

    return np.mean(auc_scores)



study_tabnet = optuna.create_study(
    direction='maximize',
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=5)
)
study_tabnet.optimize(objective_tabnet, n_trials=30, timeout=600)

print("\nBest AUC:", study_tabnet.best_value)
print("Best params:")
for k, v in study_tabnet.best_params.items():
    print(f"  {k}: {v}")

[I 2026-03-26 19:26:02,320] A new study created in memory with name: no-name-50fa5497-5a9b-4214-a727-022843aac6fc



Early stopping occurred at epoch 77 with best_epoch = 57 and best_val_0_auc = 0.65717


c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 96 with best_epoch = 76 and best_val_0_auc = 0.65546


c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


...


Early stopping occurred at epoch 56 with best_epoch = 36 and best_val_0_auc = 0.64514


c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)



Early stopping occurred at epoch 76 with best_epoch = 56 and best_val_0_auc = 0.65418


c:\Users\Marwa\anaconda3\envs\data_cleaning\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
[I 2026-03-27 00:13:16,332] Trial 0 finished with value: 0.6542776575595803 and parameters: {'n_d': 16, 'n_a': 48, 'n_steps': 4, 'gamma': 1.9474087898002561, 'lambda_sparse': 3.1937897019295946e-06, 'mask_type': 'sparsemax', 'lr': 0.0036531298640908962, 'batch_size': 256}. Best is trial 0 with value: 0.6542776575595803.



Best AUC: 0.6542776575595803
Best params:
  n_d: 16
  n_a: 48
  n_steps: 4
  gamma: 1.9474087898002561
  lambda_sparse: 3.1937897019295946e-06
  mask_type: sparsemax
  lr: 0.0036531298640908962
  batch_size: 256


Model:

In [ ]:
X_train1,X_val1,y_train1,y_val1 = train_test_split(X_train_pre,y_train,train_size=0.8,random_state=11,stratify=y_train)


clf = TabNetClassifier(
    n_d=32,              
    n_a=32,                
    n_steps=3,            
    gamma=1.3,            
    lambda_sparse=1e-3,  
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-2),
    scheduler_params={"step_size":10, "gamma":0.9},
    scheduler_fn=torch.optim.lr_scheduler.StepLR,
    mask_type='entmax',  # or sparsemax 
    seed=11
)

clf.fit(
    X_train1.values, y_train1.values,
    eval_set=[(X_val1.values, y_val1.values)],
    eval_metric=['auc'],
    max_epochs=100,
    patience=20,
    batch_size=1024,
    virtual_batch_size=128
)

Learning Curve

In [ ]:
def plot_tabnet_history(clf, metrics=['train_auc', 'val_auc']):
    history = clf.history
    
    train_auc = history['train_auc']
    val_auc   = history['val_auc']
    train_loss = history['loss']
    val_loss   = history['val_loss']

    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    axes[0].plot(train_auc,  label='Train',      color='#d65f5f')
    axes[0].plot(val_auc,    label='Validation', color='#4878d0')
    axes[0].set_title('AUC over Epochs')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, linestyle='--', alpha=0.7)

    axes[1].plot(train_loss, label='Train',      color='#d65f5f')
    axes[1].plot(val_loss,   label='Validation', color='#4878d0')
    axes[1].set_title('Loss over Epochs')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, linestyle='--', alpha=0.7)

    plt.tight_layout()
    #plt.savefig('Images_trees/LearningCurve_TabNet.png', dpi=300, bbox_inches='tight')
    plt.show()

plot_tabnet_history(clf)

Wide and Deep - neural network  - SHAPLY - Surrogate Model

In [ ]:
#try